In [23]:
!pip install pennylane

In [24]:
# Step 1: Imports & Environment Setup
import pennylane as qml
from pennylane import numpy as np
import hashlib
import time

In [25]:
# Step 2 — Quantum Device Configuration
## Finite shots simulate a real quantum measurement environment

# Two-qubit device:
# wire 0 → Server
# wire 1 → Client
dev = qml.device(
    "default.qubit",
    wires=2,
    shots=1024
)

In [26]:
# Step 3 — Classical Challenge Generator
## Prevents replay attacks by ensuring session freshness

def generate_challenge():
    """
    Generates a time-based cryptographic nonce.
    Each authentication session is unique.
    """
    nonce = str(time.time()).encode()
    return hashlib.sha256(nonce).hexdigest()

In [27]:
#Step 4 — Quantum Authentication Circuit
#Uses correlation measurement, not Bell-state collapse

@qml.qnode(dev)
def quantum_authentication_circuit(phase_angle, tamper=False):

    # Bell pair preparation
    qml.Hadamard(0)
    qml.CNOT([0, 1])

    # Tampering / MITM simulation
    if tamper:
        qml.RX(np.pi / 4, wires=1)

    # Client encodes challenge
    qml.RZ(phase_angle, wires=1)

    # Correlation measurement
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))


In [28]:
# Step 5 — Authentication Decision Logic

def authenticate_client(threshold=0.85, tamper=False):

    print("\n[+] Starting Quantum Authentication")
    challenge = generate_challenge()
    print("[+] Challenge issued")

    phase = int(challenge[:8], 16) % 360
    phase = np.deg2rad(phase)

    correlation = quantum_authentication_circuit(phase, tamper)
    fidelity = abs(correlation)

    print(f"[+] ZZ Correlation: {correlation:.3f}")
    print(f"[+] Correlation Fidelity: {fidelity:.3f}")

    if fidelity >= threshold:
        print("[✓] AUTHENTICATION SUCCESSFUL")
        return True
    else:
        print("[✗] AUTHENTICATION FAILED — TAMPERING DETECTED")
        return False

In [29]:
# Step 6: Demo 1: Legitimate Client Authentication
## Expected outcome: Authentication succeeds

# Legitimate client (no tampering)
authenticate_client(tamper=False)


[+] Starting Quantum Authentication
[+] Challenge issued
[+] ZZ Correlation: 1.000
[+] Correlation Fidelity: 1.000
[✓] AUTHENTICATION SUCCESSFUL


True

In [30]:
# Step 7 — Demo 2: Interception / MITM Attack Simulation
## Expected outcome: Authentication fails

# Simulated attacker intercepts or measures the qubit
authenticate_client(tamper=True)


[+] Starting Quantum Authentication
[+] Challenge issued
[+] ZZ Correlation: 0.697
[+] Correlation Fidelity: 0.697
[✗] AUTHENTICATION FAILED — TAMPERING DETECTED


False

In [ ]:
# What This Demonstrates in the Project (Quantum Layer – Corrected)

#• No passwords or stored credentials; authentication relies on shared quantum correlations  
#• Authentication is physics-based, verified via entanglement correlations rather than state identity  
#• Phase-based challenge encoding preserves entanglement while preventing replay attacks  
#• Legitimate clients maintain strong ⟨Z ⊗ Z⟩ correlations close to +1  
#• MITM or interception attacks disturb entanglement, reducing measurable correlations  
#• Detection is statistical and measurement-based, consistent with real quantum systems  
#• Finite-shot measurements reflect practical quantum hardware constraints  
#• Layer is directly composable with Diffie–Hellman + HMAC for intent verification